In [ ]:
#1. Implement a model-based intelligent agent capable of navigating a grid environment with obstacles using internal state representation.
class ModelBasedAgent:
    def __init__(self, grid, start, goal):
        self.grid = grid
        self.position = start
        self.goal = goal

        # Internal state: stores cells already known to be blocked
        self.known_obstacles = set()

        # Movement directions: Up, Down, Left, Right
        self.actions = [
            (-1, 0),  # Up
            (1, 0),   # Down
            (0, -1),  # Left
            (0, 1)    # Right
        ]

    def perceive(self):
        """Observe obstacles around the current position."""
        r, c = self.position

        for dr, dc in self.actions:
            nr, nc = r + dr, c + dc

            if (0 <= nr < len(self.grid) and
                0 <= nc < len(self.grid[0])):

                if self.grid[nr][nc] == 1:
                    self.known_obstacles.add((nr, nc))

    def choose_action(self):
        """Choose the best safe move toward the goal."""
        r, c = self.position
        gr, gc = self.goal

        possible_moves = []

        for dr, dc in self.actions:
            nr, nc = r + dr, c + dc

            if (0 <= nr < len(self.grid) and
                0 <= nc < len(self.grid[0])):

                if (nr, nc) not in self.known_obstacles:
                    distance = abs(nr - gr) + abs(nc - gc)
                    possible_moves.append(
                        (distance, (nr, nc))
                    )

        if not possible_moves:
            return None

        # Select the move closest to the goal
        possible_moves.sort()
        return possible_moves[0][1]

    def navigate(self):
        """Navigate from start to goal."""
        path = [self.position]

        while self.position != self.goal:

            # Perceive the environment
            self.perceive()

            # Select next action
            next_position = self.choose_action()

            if next_position is None:
                print("Agent is blocked!")
                return path

            self.position = next_position
            path.append(self.position)

        return path


# 0 = Free cell, 1 = Obstacle
grid = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0]
]

start = (0, 0)
goal = (4, 4)

agent = ModelBasedAgent(grid, start, goal)

path = agent.navigate()

print("Path followed by agent:")
print(path)

print("\nKnown obstacles:")
print(agent.known_obstacles)

Path followed by agent:
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]

Known obstacles:
{(1, 2), (1, 1), (2, 3), (1, 3)}


In [ ]:
#2. Implement A* Search for 8 puzzle problem only python program

import heapq

# Goal state
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

# Manhattan Distance Heuristic
def heuristic(state):
    distance = 0

    for i in range(9):
        if state[i] != 0:
            goal_pos = GOAL.index(state[i])
            distance += abs(i // 3 - goal_pos // 3)
            distance += abs(i % 3 - goal_pos % 3)

    return distance


# Generate possible next states
def get_neighbors(state):
    neighbors = []
    zero = state.index(0)

    row, col = zero // 3, zero % 3

    moves = [
        (-1, 0),  # Up
        (1, 0),   # Down
        (0, -1),  # Left
        (0, 1)    # Right
    ]

    for dr, dc in moves:
        new_row = row + dr
        new_col = col + dc

        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_zero = new_row * 3 + new_col

            new_state = list(state)
            new_state[zero], new_state[new_zero] = \
                new_state[new_zero], new_state[zero]

            neighbors.append(tuple(new_state))

    return neighbors


# A* Search
def a_star(start):
    open_list = []

    # (f, g, state, path)
    heapq.heappush(
        open_list,
        (heuristic(start), 0, start, [start])
    )

    visited = set()

    while open_list:
        f, g, current, path = heapq.heappop(open_list)

        if current in visited:
            continue

        visited.add(current)

        if current == GOAL:
            return path

        for neighbor in get_neighbors(current):
            if neighbor not in visited:
                new_g = g + 1
                new_f = new_g + heuristic(neighbor)

                heapq.heappush(
                    open_list,
                    (new_f, new_g, neighbor, path + [neighbor])
                )

    return None


# Print puzzle
def print_puzzle(state):
    for i in range(0, 9, 3):
        print(state[i:i+3])
    print()


# Input initial state
start = (
    1, 2, 3,
    4, 0, 6,
    7, 5, 8
)

# Solve using A*
solution = a_star(start)

if solution:
    print("Solution found!")
    print("Number of moves:", len(solution) - 1)
    print("\nSteps:\n")

    for step, state in enumerate(solution):
        print("Step", step)
        print_puzzle(state)
else:
    print("No solution exists.")

Solution found!
Number of moves: 2

Steps:

Step 0
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 1
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Step 2
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)



In [ ]:
#3. Implement Tower of Hanoi by State Space Search

from collections import deque

def tower_of_hanoi(n):
    start = tuple([tuple(range(n, 0, -1)), (), ()])
    goal = ((), (), tuple(range(n, 0, -1)))

    queue = deque([(start, [])])
    visited = {start}

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path

        # Generate all possible moves
        for source in range(3):
            if not state[source]:
                continue

            for destination in range(3):
                if source == destination:
                    continue

                # Move only if destination is empty
                # or top disk is larger
                if (not state[destination] or
                    state[source][-1] < state[destination][-1]):

                    new_state = [list(peg) for peg in state]
                    disk = new_state[source].pop()
                    new_state[destination].append(disk)

                    new_state = tuple(tuple(peg) for peg in new_state)

                    if new_state not in visited:
                        visited.add(new_state)

                        move = (
                            f"Move disk {disk} "
                            f"from {source + 1} to {destination + 1}"
                        )

                        queue.append(
                            (new_state, path + [move])
                        )

    return None


# Number of disks
n = 3

solution = tower_of_hanoi(n)

print("Tower of Hanoi using State Space Search")
print("----------------------------------------")

for i, move in enumerate(solution, 1):
    print(f"Step {i}: {move}")

print("\nTotal moves:", len(solution))

Tower of Hanoi using State Space Search
----------------------------------------
Step 1: Move disk 1 from 1 to 3
Step 2: Move disk 2 from 1 to 2
Step 3: Move disk 1 from 3 to 2
Step 4: Move disk 3 from 1 to 3
Step 5: Move disk 1 from 2 to 1
Step 6: Move disk 2 from 2 to 3
Step 7: Move disk 1 from 1 to 3

Total moves: 7


In [ ]:
#4. Implement a Sudoku solver using backtracking and constraint propagation techniques.


def find_empty(board):
    # Constraint propagation: choose the cell
    # with minimum possible values (MRV)
    best_cell = None
    best_options = None

    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                options = get_options(board, r, c)

                if best_options is None or len(options) < len(best_options):
                    best_cell = (r, c)
                    best_options = options

    if best_cell is None:
        return None, None

    return best_cell, best_options


def get_options(board, row, col):
    used = set()

    # Row constraint
    for num in board[row]:
        if num != 0:
            used.add(num)

    # Column constraint
    for r in range(9):
        if board[r][col] != 0:
            used.add(board[r][col])

    # 3x3 box constraint
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    for r in range(start_row, start_row + 3):
        for c in range(start_col, start_col + 3):
            if board[r][c] != 0:
                used.add(board[r][c])

    return [num for num in range(1, 10) if num not in used]


def solve(board):
    cell, options = find_empty(board)

    # No empty cell means Sudoku is solved
    if cell is None:
        return True

    row, col = cell

    # Backtracking with constraint propagation
    for num in options:
        board[row][col] = num

        if solve(board):
            return True

        # Undo choice if it leads to failure
        board[row][col] = 0

    return False


def print_board(board):
    for row in board:
        print(" ".join(map(str, row)))


# 0 represents an empty cell
board = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],

    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],

    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]


print("Sudoku Before Solving:")
print_board(board)

if solve(board):
    print("\nSudoku After Solving:")
    print_board(board)
else:
    print("\nNo solution exists.")

Sudoku Before Solving:
5 3 0 0 7 0 0 0 0
6 0 0 1 9 5 0 0 0
0 9 8 0 0 0 0 6 0
8 0 0 0 6 0 0 0 3
4 0 0 8 0 3 0 0 1
7 0 0 0 2 0 0 0 6
0 6 0 0 0 0 2 8 0
0 0 0 4 1 9 0 0 5
0 0 0 0 8 0 0 7 9

Sudoku After Solving:
5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


In [ ]:
#B1. Use BFS to solve a planning problem. Examples can be 8 puzzle problem, Robot path planning, Blocks world etc.

from collections import deque

# Grid: 0 = free cell, 1 = obstacle
grid = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0]
]

start = (0, 0)
goal = (4, 4)

# BFS for robot path planning
def bfs(start, goal):
    queue = deque([(start, [start])])
    visited = {start}

    directions = [
        (-1, 0),  # Up
        (1, 0),   # Down
        (0, -1),  # Left
        (0, 1)    # Right
    ]

    while queue:
        current, path = queue.popleft()

        # Goal reached
        if current == goal:
            return path

        r, c = current

        for dr, dc in directions:
            nr = r + dr
            nc = c + dc

            # Check valid position
            if (0 <= nr < len(grid) and
                0 <= nc < len(grid[0]) and
                grid[nr][nc] == 0 and
                (nr, nc) not in visited):

                visited.add((nr, nc))
                queue.append(((nr, nc), path + [(nr, nc)]))

    return None


# Solve planning problem
path = bfs(start, goal)

if path:
    print("Robot Path using BFS:")
    for i, position in enumerate(path):
        print("Step", i, ":", position)

    print("\nTotal steps:", len(path) - 1)
else:
    print("No path found.")

Robot Path using BFS:
Step 0 : (0, 0)
Step 1 : (1, 0)
Step 2 : (2, 0)
Step 3 : (3, 0)
Step 4 : (4, 0)
Step 5 : (4, 1)
Step 6 : (4, 2)
Step 7 : (4, 3)
Step 8 : (4, 4)

Total steps: 8


In [ ]:
#B2. Implement Backtracking search to solve the 8-Queens constraint satisfaction problem.

# 8-Queens Problem using Backtracking

N = 8
board = [-1] * N

def is_safe(row, col):
    for prev_row in range(row):
        prev_col = board[prev_row]

        # Same column
        if prev_col == col:
            return False

        # Same diagonal
        if abs(prev_row - row) == abs(prev_col - col):
            return False

    return True


def solve(row):
    # All queens placed
    if row == N:
        return True

    # Try every column
    for col in range(N):
        if is_safe(row, col):
            board[row] = col

            if solve(row + 1):
                return True

            # Backtrack
            board[row] = -1

    return False


def print_board():
    for row in range(N):
        for col in range(N):
            if board[row] == col:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()


if solve(0):
    print("Solution for 8-Queens:")
    print_board()
else:
    print("No solution exists.")

Solution for 8-Queens:
Q . . . . . . . 
. . . . Q . . . 
. . . . . . . Q 
. . . . . Q . . 
. . Q . . . . . 
. . . . . . Q . 
. Q . . . . . . 
. . . Q . . . . 


In [ ]:
#B3. Design an intelligent agent that guesses a number using feedback (higher/lower). 


# Intelligent Agent for Number Guessing
# Feedback: Higher / Lower
# Accepted inputs: h, l, c

def intelligent_agent(low, high):
    print(f"Think of a number between {low} and {high}.")
    input("Press Enter when ready...")

    attempts = 0

    while low <= high:
        guess = (low + high) // 2
        attempts += 1

        print(f"\nAgent guesses: {guess}")

        feedback = input("Enter feedback (H:Higher/ L:Lower/ C:Correct): ").strip().lower()

        if feedback in ("h", "higher","H"):
            print(f"Feedback: higher")
            low = guess + 1

        elif feedback in ("l", "lower","L"):
            print(f"Feedback: lower")
            high = guess - 1

        elif feedback in ("c", "correct","C"):
            print(f"Feedback: correct")
            print("\nNumber found!")
            print("Number:", guess)
            print("Attempts:", attempts)
            return

        else:
            print("Invalid feedback! Enter h, l, or c.")
            continue

    print("No valid number found.")


# Start the intelligent agent
intelligent_agent(1, 100)


Think of a number between 1 and 100.

Agent guesses: 50
Invalid feedback! Enter h, l, or c.

Agent guesses: 50
Feedback: lower

Agent guesses: 25
Feedback: lower

Agent guesses: 12
Feedback: higher

Agent guesses: 18


In [1]:
#B4. Implement Goal Stack Planning for solving the Blocks World problem and achieve a specified goal configuration.


# Initial configuration
initial = {
    "A": "Table",
    "B": "A",
    "C": "Table"
}

# Goal configuration
goal = {
    "A": "Table",
    "B": "C",
    "C": "Table"
}


def display(state):
    print("\nCurrent State:")
    for block, position in state.items():
        if position == "Table":
            print(block, "is on Table")
        else:
            print(block, "is on", position)


def clear_block(state, block, plan):
    """Move everything on top of the given block to the table."""
    for item, position in list(state.items()):
        if position == block and item != block:
            clear_block(state, item, plan)
            state[item] = "Table"
            plan.append(f"Move {item} to Table")


def goal_stack_planning(initial_state, goal_state):
    state = initial_state.copy()
    plan = []

    def achieve(block):
        if state[block] == goal_state[block]:
            return

        # Clear the target if something is already on it.
        if goal_state[block] != "Table":
            target = goal_state[block]
            for item, position in list(state.items()):
                if item != block and position == target:
                    achieve(item)
                    state[item] = "Table"
                    plan.append(f"Move {item} to Table")

        # Clear the block itself if another block is on top of it.
        clear_block(state, block, plan)

        # Move the block to its goal position.
        state[block] = goal_state[block]
        plan.append(f"Move {block} to {goal_state[block]}")

    # Achieve each goal in reverse order to keep the stack valid.
    for block in reversed(list(goal_state.keys())):
        achieve(block)

    return state, plan


final_state, plan = goal_stack_planning(initial, goal)

print("Initial Configuration:")
display(initial)

print("\nGoal Configuration:")
display(goal)

print("\nPlan:")
for i, action in enumerate(plan, 1):
    print("Step", i, ":", action)

print("\nFinal Configuration:")
display(final_state)


Initial Configuration:

Current State:
A is on Table
B is on A
C is on Table

Goal Configuration:

Current State:
A is on Table
B is on C
C is on Table

Plan:
Step 1 : Move B to C

Final Configuration:

Current State:
A is on Table
B is on C
C is on Table
